<a href="https://colab.research.google.com/github/AnjanPayra/MEM-FET-Essential-protein-prediction-using-membership-feature-and-machine-learning-approach/blob/main/MEM_FET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, mean_squared_error, precision_score,
                              recall_score, f1_score, roc_auc_score, matthews_corrcoef,
                              confusion_matrix)
from xgboost import XGBClassifier

UPLOAD_DIR = "/content"
OUT_DIR = "/mnt/user-data/outputs"

NETWORK_FILES = {
    "YDIP": f"{UPLOAD_DIR}/YDIP.txt",
    "YHQ": f"{UPLOAD_DIR}/YHQ.txt",
    "YMBD": f"{UPLOAD_DIR}/YMBD.txt",
    "YMIPS": f"{UPLOAD_DIR}/YMIPS.txt",
}
ESSENTIAL_XLSX = "/content/Essential.xlsx"
RANDOM_STATE = 42
FEATURE_NAMES = ["ECC", "CC", "GO_Nb", "SL_Nb"]

In [15]:
# ---------------------------------------------------------------------------
# 1. Load network
# ---------------------------------------------------------------------------
def load_network(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    cols = list(df.columns)
    df = df.rename(columns={cols[0]: "Protein1", cols[1]: "Protein2"})
    df["Protein1"] = df["Protein1"].astype(str).str.strip()
    df["Protein2"] = df["Protein2"].astype(str).str.strip()
    df = df.dropna(subset=["Protein1", "Protein2"])
    G = nx.Graph()
    G.add_edges_from(zip(df["Protein1"], df["Protein2"]))
    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


In [3]:

# ---------------------------------------------------------------------------
# 2. Feature generation: ECC, CC, GO_Nb, SL_Nb
# ---------------------------------------------------------------------------
def compute_ecc(G):
    degree = dict(G.degree())
    ecc_uv = {}
    for u, v in G.edges():
        z_uv = len(list(nx.common_neighbors(G, u, v)))
        denom = max(degree[u], degree[v])
        ecc_uv[(u, v)] = z_uv / denom if denom > 0 else 0.0
    ecc_u = {n: 0.0 for n in G.nodes()}
    for (u, v), val in ecc_uv.items():
        ecc_u[u] += val
        ecc_u[v] += val
    return ecc_u


def compute_cc(G):
    # CC_i = 2*triangles_i / (deg_i*(deg_i-1))  ==  nx.clustering
    return nx.clustering(G)


def compute_go_nb(G, go_annotations=None):
    if not go_annotations:
        return {n: 0.0 for n in G.nodes()}
    go_nb_u = {n: 0.0 for n in G.nodes()}
    for u, v in G.edges():
        gu, gv = go_annotations.get(u, set()), go_annotations.get(v, set())
        if not gu or not gv:
            continue
        common = set(G[u]) & set(G[v])
        p = sum(len((gu & go_annotations.get(t, set())) | (gv & go_annotations.get(t, set())))
                for t in common)
        q = min(len(gu), len(gv))
        val = p / q if q > 0 else 0.0
        go_nb_u[u] += val
        go_nb_u[v] += val
    return go_nb_u


def compute_sl_nb(G, sl_annotations=None):
    if not sl_annotations:
        return {n: 0.0 for n in G.nodes()}
    sl_nb_u = {n: 0.0 for n in G.nodes()}
    for u, v in G.edges():
        su, sv = sl_annotations.get(u, set()), sl_annotations.get(v, set())
        if not su or not sv:
            continue
        common = set(G[u]) & set(G[v])
        r = len({t for t in common if sl_annotations.get(t, set())}) ** 2
        s = len(su) * len(sv)
        val = r / s if s > 0 else 0.0
        sl_nb_u[u] += val
        sl_nb_u[v] += val
    return sl_nb_u



In [4]:
# ---------------------------------------------------------------------------
# 3. MEM-FET(phi) = ECC(phi) + MAX(CC(phi),GO_Nb(phi)) / MAX(ECC(phi),CC(phi),GO_Nb(phi))
# ---------------------------------------------------------------------------
def compute_mem_fet(ecc, cc, go_nb):
    mem_fet = {}
    for n in ecc:
        e, c, g = ecc[n], cc.get(n, 0.0), go_nb.get(n, 0.0)
        denom = max(e, c, g)
        mem_fet[n] = e + (max(c, g) / denom if denom > 0 else 0.0)
    return mem_fet


In [7]:
# ---------------------------------------------------------------------------
# 4. Feature ranking: RandomForest, XGBoost, LogisticRegression, RFE
# ---------------------------------------------------------------------------
def rank_features(X, y):
    results = {}

    rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced")
    rf.fit(X, y)
    results["RandomForest"] = dict(zip(FEATURE_NAMES, rf.feature_importances_))

    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                         eval_metric="logloss", random_state=RANDOM_STATE,
                         scale_pos_weight=(y == 0).sum() / max((y == 1).sum(), 1))
    xgb.fit(X, y)
    results["XGBoost"] = dict(zip(FEATURE_NAMES, xgb.feature_importances_))

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    lr.fit(Xs, y)
    lr_importance = np.abs(lr.coef_[0])
    lr_importance = lr_importance / lr_importance.sum() if lr_importance.sum() > 0 else lr_importance
    results["LogisticRegression"] = dict(zip(FEATURE_NAMES, lr_importance))

    rfe_estimator = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    rfe = RFE(rfe_estimator, n_features_to_select=1, step=1)
    rfe.fit(Xs, y)
    # convert ranking (1=best) to a normalized importance-like score (higher=better)
    max_rank = rfe.ranking_.max()
    rfe_score = (max_rank + 1 - rfe.ranking_).astype(float)
    rfe_score = rfe_score / rfe_score.sum()
    results["RFE"] = dict(zip(FEATURE_NAMES, rfe_score))

    rank_df = pd.DataFrame(results)
    rank_df["Average"] = rank_df.mean(axis=1)
    rank_df = rank_df.sort_values("Average", ascending=False)
    return rank_df




In [6]:
# ---------------------------------------------------------------------------
# 5. Gold standard + dataset formation
# ---------------------------------------------------------------------------
def load_gold_standard(path):
    xls = pd.ExcelFile(path)
    ess_sheet = [s for s in xls.sheet_names if "non" not in s.lower() and "essential" in s.lower()][0]
    non_sheet = [s for s in xls.sheet_names if "non" in s.lower() and "essential" in s.lower()][0]
    essential = set(pd.read_excel(path, sheet_name=ess_sheet, header=None)[0].dropna().astype(str).str.strip())
    nonessential = set(pd.read_excel(path, sheet_name=non_sheet, header=None)[0].dropna().astype(str).str.strip())
    return essential, nonessential

In [8]:

# ---------------------------------------------------------------------------
# 6. Ensemble evaluation (Accuracy + MSE + extra metrics)
# ---------------------------------------------------------------------------
def evaluate_ensemble(X, y, feature_cols):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced")
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, eval_metric="logloss",
                         random_state=RANDOM_STATE,
                         scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1))
    lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)

    ensemble = VotingClassifier(estimators=[("rf", rf), ("xgb", xgb), ("lr", lr)], voting="soft")
    ensemble.fit(X_train_s, y_train)

    y_pred = ensemble.predict(X_test_s)
    y_prob = ensemble.predict_proba(X_test_s)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    metrics = dict(
        Accuracy=accuracy_score(y_test, y_pred),
        MSE=mean_squared_error(y_test, y_prob),
        Precision=precision_score(y_test, y_pred, zero_division=0),
        Recall=recall_score(y_test, y_pred, zero_division=0),
        F1=f1_score(y_test, y_pred, zero_division=0),
        ROC_AUC=roc_auc_score(y_test, y_prob) if len(set(y_test)) > 1 else float("nan"),
        MCC=matthews_corrcoef(y_test, y_pred),
        TP=tp, FP=fp, FN=fn, TN=tn,
        n_train=len(y_train), n_test=len(y_test),
    )
    return metrics, ensemble, scaler



In [9]:
# ---------------------------------------------------------------------------
# 3-point threshold (unsupervised check, using MEM-FET as the score)
# ---------------------------------------------------------------------------
def three_point_threshold(scores, k):
    arr = np.array(list(scores.values()), dtype=float)
    mean, std = arr.mean(), arr.std()
    return mean + k * std * (1 - 1 / (1 + std ** 2))


def confusion_metrics(predicted_set, essential_set, nonessential_set, universe):
    labeled = universe & (essential_set | nonessential_set)
    TP = sum(1 for n in labeled if n in predicted_set and n in essential_set)
    FP = sum(1 for n in labeled if n in predicted_set and n in nonessential_set)
    FN = sum(1 for n in labeled if n not in predicted_set and n in essential_set)
    TN = sum(1 for n in labeled if n not in predicted_set and n in nonessential_set)

    def sd(a, b):
        return a / b if b else float("nan")

    sens, spec = sd(TP, TP + FN), sd(TN, TN + FP)
    ppv, npv = sd(TP, TP + FP), sd(TN, TN + FN)
    f1 = sd(2 * ppv * sens, ppv + sens) if not (math.isnan(ppv) or math.isnan(sens)) else float("nan")
    acc = sd(TP + TN, TP + TN + FP + FN)
    denom = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = sd(TP * TN - FP * FN, denom) if denom else float("nan")
    return dict(TP=TP, FP=FP, FN=FN, TN=TN, Sensitivity=sens, Specificity=spec,
                PPV=ppv, NPV=npv, F1=f1, Accuracy=acc, MCC=mcc, n_labeled=len(labeled))


In [17]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
import os
def main():
    essential_set, nonessential_set = load_gold_standard(ESSENTIAL_XLSX)
    print(f"Gold standard: {len(essential_set)} essential, {len(nonessential_set)} non-essential\n")

    # Fill these in if you obtain real GO / subcellular-localization data:
    go_annotations = None   # dict: protein -> set(GO terms)
    sl_annotations = None   # dict: protein -> set(compartments)

    os.makedirs(OUT_DIR, exist_ok=True)
    writer = pd.ExcelWriter(f"{OUT_DIR}/mem_fet_results.xlsx", engine="openpyxl")
    all_rank_dfs = {}
    all_ensemble_metrics = []
    all_threshold_metrics = []

    for name, path in NETWORK_FILES.items():
        print(f"\n========== {name} ==========")
        G = load_network(path)
        universe = set(G.nodes())
        print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

        # ---- Stage 1: feature generation ----
        ecc = compute_ecc(G)
        cc = compute_cc(G)
        go_nb = compute_go_nb(G, go_annotations)
        sl_nb = compute_sl_nb(G, sl_annotations)

        feat_df = pd.DataFrame({
            "Protein": list(G.nodes()),
            "ECC": [ecc[n] for n in G.nodes()],
            "CC": [cc[n] for n in G.nodes()],
            "GO_Nb": [go_nb[n] for n in G.nodes()],
            "SL_Nb": [sl_nb[n] for n in G.nodes()],
        })
        feat_df["Label"] = feat_df["Protein"].apply(
            lambda n: "Essential" if n in essential_set else ("NonEssential" if n in nonessential_set else "Unlabeled"))

        labeled_df = feat_df[feat_df["Label"] != "Unlabeled"].copy()
        labeled_df["y"] = (labeled_df["Label"] == "Essential").astype(int)
        X_all = labeled_df[FEATURE_NAMES].values
        y_all = labeled_df["y"].values
        print(f"Labeled proteins in network: {len(labeled_df)} "
              f"({labeled_df['y'].sum()} essential / {(labeled_df['y']==0).sum()} non-essential)")

        # ---- Stage 2: rank features ----
        rank_df = rank_features(X_all, y_all)
        all_rank_dfs[name] = rank_df
        print("Feature ranking (avg importance across RF/XGB/LR/RFE):")
        print(rank_df.round(4).to_string())

        # ---- Stage 3: MEM-FET ----
        mem_fet = compute_mem_fet(ecc, cc, go_nb)
        feat_df["MEM_FET"] = feat_df["Protein"].map(mem_fet)
        labeled_df["MEM_FET"] = labeled_df["Protein"].map(mem_fet)

        # ---- Stage 4: dataset formation (ranked features + MEM-FET) ----
        model_features = FEATURE_NAMES + ["MEM_FET"]
        X_model = labeled_df[model_features].values
        y_model = labeled_df["y"].values

        # ---- Stage 5: ensemble evaluation ----
        metrics, ensemble, scaler = evaluate_ensemble(X_model, y_model, model_features)
        metrics["Network"] = name
        all_ensemble_metrics.append(metrics)
        print(f"Ensemble (RF+XGB+LR soft-voting) on held-out 25%: "
              f"Accuracy={metrics['Accuracy']:.3f}  MSE={metrics['MSE']:.4f}  "
              f"Precision={metrics['Precision']:.3f}  Recall={metrics['Recall']:.3f}  "
              f"F1={metrics['F1']:.3f}  ROC-AUC={metrics['ROC_AUC']:.3f}  MCC={metrics['MCC']:.3f}")

        # ---- Unsupervised 3-point-threshold check using MEM-FET ----
        for k in [1, 2, 3]:
            thr = three_point_threshold(mem_fet, k)
            predicted = {n for n, v in mem_fet.items() if v > thr}
            tm = confusion_metrics(predicted, essential_set, nonessential_set, universe)
            tm.update(Network=name, K=k, Threshold=round(thr, 4), NumPredicted=len(predicted))
            all_threshold_metrics.append(tm)

        # ---- write per-network sheets ----
        feat_df.sort_values("MEM_FET", ascending=False).to_excel(writer, sheet_name=f"{name}_features", index=False)
        rank_df.to_excel(writer, sheet_name=f"{name}_feat_rank")

    ensemble_summary = pd.DataFrame(all_ensemble_metrics)
    threshold_summary = pd.DataFrame(all_threshold_metrics)
    ensemble_summary.to_excel(writer, sheet_name="Ensemble_summary", index=False)
    threshold_summary.to_excel(writer, sheet_name="MEMFET_threshold_summary", index=False)
    writer.close()

    # ---- comparison plot: ensemble accuracy & MSE per network ----
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].bar(ensemble_summary["Network"], ensemble_summary["Accuracy"], color="#4C72B0")
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Ensemble classifier Accuracy (held-out test set)")
    axes[0].set_ylabel("Accuracy")
    axes[1].bar(ensemble_summary["Network"], ensemble_summary["MSE"], color="#DD8452")
    axes[1].set_title("Ensemble classifier MSE (predicted prob vs label)")
    axes[1].set_ylabel("MSE")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/mem_fet_ensemble_performance.png", dpi=150)
    plt.close()

    print("\n\n=== ENSEMBLE SUMMARY (all networks) ===")
    print(ensemble_summary[["Network", "Accuracy", "MSE", "Precision", "Recall", "F1", "ROC_AUC", "MCC"]]
          .round(4).to_string(index=False))

    print("\n=== MEM-FET 3-point-threshold SUMMARY (all networks) ===")
    print(threshold_summary[["Network", "K", "Threshold", "NumPredicted", "Sensitivity",
                              "Specificity", "PPV", "Accuracy", "MCC"]].round(4).to_string(index=False))

    return ensemble_summary, threshold_summary, all_rank_dfs


if __name__ == "__main__":
    main()

Gold standard: 1285 essential, 4394 non-essential


========== YDIP ==========
Nodes: 5093, Edges: 24743
Labeled proteins in network: 4758 (1167 essential / 3591 non-essential)
Feature ranking (avg importance across RF/XGB/LR/RFE):
       RandomForest  XGBoost  LogisticRegression  RFE  Average
ECC          0.6411   0.6897              0.7561  0.4   0.6217
CC           0.3589   0.3103              0.2439  0.3   0.3033
GO_Nb        0.0000   0.0000              0.0000  0.2   0.0500
SL_Nb        0.0000   0.0000              0.0000  0.1   0.0250
Ensemble (RF+XGB+LR soft-voting) on held-out 25%: Accuracy=0.714  MSE=0.2085  Precision=0.401  Recall=0.332  F1=0.363  ROC-AUC=0.605  MCC=0.183

========== YHQ ==========
Nodes: 4743, Edges: 22665
Labeled proteins in network: 4361 (1108 essential / 3253 non-essential)
Feature ranking (avg importance across RF/XGB/LR/RFE):
       RandomForest  XGBoost  LogisticRegression  RFE  Average
CC           0.3524   0.2862              0.9703  0.4   0.5022
ECC